# Google Gemini - Ejemplos de uso

Este notebook muestra cómo usar la API de Google Gemini para crear aplicaciones con IA.

**Modelos disponibles:**
- `gemini-1.5-pro` - El más capaz
- `gemini-1.5-flash` - Más rápido y económico
- `gemini-2.0-flash` - Última generación

**Documentación:** https://ai.google.dev/gemini-api/docs  
**Obtén tu clave en:** https://aistudio.google.com/app/apikey

## 1. Instalación de dependencias

In [8]:
%pip install google-genai python-dotenv

Note: you may need to restart the kernel to use updated packages.


## 2. Configuración de la clave de API

In [9]:
import os
from pathlib import Path
from dotenv import load_dotenv
from google import genai

# Cargar variables de entorno desde .env
load_dotenv(Path('..') / '.env')

# Verificar que la clave esté configurada
api_key = os.getenv('GEMINI_API_KEY')
if api_key:
    print(f'✅ Clave de API configurada: {api_key[:8]}...')
else:
    print('❌ GEMINI_API_KEY no encontrada. Configura tu archivo .env')

# Crear cliente Gemini con la clave del entorno
client = genai.Client(api_key=api_key)

def elegir_modelo_disponible(client, candidatos):
    disponibles = []
    for m in client.models.list():
        nombre = getattr(m, 'name', '')
        if 'generateContent' in getattr(m, 'supported_actions', []):
            disponibles.append(nombre)

    for candidato in candidatos:
        if candidato in disponibles:
            return candidato, disponibles

    if disponibles:
        return disponibles[0], disponibles

    raise RuntimeError('No hay modelos de Gemini disponibles para generateContent en esta cuenta.')

MODELO_FLASH, _modelos_disponibles = elegir_modelo_disponible(
    client,
    ['gemini-2.5-flash', 'gemini-2.0-flash', 'gemini-1.5-flash']
    )
MODELO_PRO, _ = elegir_modelo_disponible(
    client,
    ['gemini-2.5-pro', 'gemini-1.5-pro', MODELO_FLASH]
    )

print(f'✅ Modelo de texto seleccionado: {MODELO_FLASH}')
print(f'✅ Modelo avanzado seleccionado: {MODELO_PRO}')

✅ Clave de API configurada: AIzaSyA8...
✅ Modelo de texto seleccionado: models/gemini-2.5-flash
✅ Modelo avanzado seleccionado: models/gemini-2.5-flash


## 3. Ejemplo básico: Generación de texto

In [10]:
respuesta = client.models.generate_content(
    model=MODELO_FLASH,
    contents='¿Qué es el machine learning? Explícalo en 3 puntos clave.'
)

print('Respuesta:')
print(respuesta.text)

Respuesta:
El Machine Learning (Aprendizaje Automático) es una rama de la Inteligencia Artificial que permite a los sistemas **aprender de los datos** y **mejorar su rendimiento** en una tarea específica, sin ser programados explícitamente para cada posible escenario.

Aquí te lo explico en 3 puntos clave:

1.  **Aprendizaje a través de Datos (No Programación Explícita):**
    A diferencia de la programación tradicional, donde se le dan al ordenador instrucciones paso a paso para cada tarea, en Machine Learning, la máquina "aprende" a partir de grandes volúmenes de datos. Se le presentan ejemplos (datos de entrenamiento), y el sistema identifica patrones, relaciones y estructuras dentro de esos datos por sí mismo, sin que se le digan las reglas específicas de antemano.

2.  **Objetivo: Predecir y Tomar Decisiones:**
    El propósito principal del Machine Learning es desarrollar algoritmos capaces de analizar los patrones detectados en los datos de entrenamiento para luego aplicar ese c

## 4. Ejemplo: Chat con historial

In [11]:
chat = client.chats.create(model=MODELO_FLASH)

respuesta1 = chat.send_message('¿Cuáles son los principales frameworks de Python para IA?')
print('Gemini:', respuesta1.text)

respuesta2 = chat.send_message('¿Cuál me recomiendas para comenzar?')
print('Gemini:', respuesta2.text)

Gemini: Python es el lenguaje de programación dominante en el campo de la Inteligencia Artificial, el Machine Learning y el Deep Learning debido a su simplicidad, su vasta comunidad y su increíble ecosistema de bibliotecas y frameworks.

Aquí te presento los principales frameworks y bibliotecas de Python para IA, categorizados por su uso principal:

---

### **1. Fundamentos de Datos y Computación Numérica**

Estas no son frameworks de IA per se, pero son la base fundamental sobre la que se construyen todos los demás:

*   **NumPy:** La biblioteca fundamental para la computación numérica en Python. Proporciona un objeto de array N-dimensional de alto rendimiento y herramientas para trabajar con estos arrays. Es la base para casi todas las bibliotecas de ML.
*   **Pandas:** Esencial para la manipulación y el análisis de datos. Ofrece estructuras de datos como DataFrames que facilitan el manejo de datos tabulares, la limpieza, el filtrado y la preparación para modelos de IA.
*   **SciPy:

## 5. Ejemplo: Iniciar chat con historial previo

In [ ]:
historial = [
    {'role': 'user', 'parts': [{'text': '¿Qué es Python?'}]},
    {'role': 'model', 'parts': [{'text': 'Python es un lenguaje de programación de alto nivel.'}]},
    {'role': 'user', 'parts': [{'text': '¿Cuáles son sus principales ventajas para IA?'}]},
]

respuesta = client.models.generate_content(
    model=MODELO_FLASH,
    contents=historial
)

print('Gemini:', respuesta.text)

## 6. Ejemplo: Generación de código

In [ ]:
from google.genai import types

respuesta = client.models.generate_content(
    model=MODELO_PRO,
    contents='Crea un endpoint de FastAPI que reciba un texto y devuelva su análisis de sentimiento.',
    config=types.GenerateContentConfig(
        system_instruction='Eres un experto en desarrollo web con FastAPI y Python. Genera siempre código limpio, bien comentado y siguiendo las mejores prácticas.'
    ),
)

print(respuesta.text)

## 7. Listar modelos disponibles

In [12]:
print('Modelos disponibles de Gemini (generateContent):')
for modelo in _modelos_disponibles:
    if 'gemini' in modelo.lower():
        print(f'  - {modelo}')

Modelos disponibles de Gemini (generateContent):
  - models/gemini-2.5-flash
  - models/gemini-2.5-pro
  - models/gemini-2.0-flash
  - models/gemini-2.0-flash-001
  - models/gemini-2.0-flash-lite-001
  - models/gemini-2.0-flash-lite
  - models/gemini-2.5-flash-preview-tts
  - models/gemini-2.5-pro-preview-tts
  - models/gemini-flash-latest
  - models/gemini-flash-lite-latest
  - models/gemini-pro-latest
  - models/gemini-2.5-flash-lite
  - models/gemini-2.5-flash-image
  - models/gemini-2.5-flash-lite-preview-09-2025
  - models/gemini-3-pro-preview
  - models/gemini-3-flash-preview
  - models/gemini-3.1-pro-preview
  - models/gemini-3.1-pro-preview-customtools
  - models/gemini-3.1-flash-lite-preview
  - models/gemini-3-pro-image-preview
  - models/gemini-3.1-flash-image-preview
  - models/gemini-robotics-er-1.5-preview
  - models/gemini-2.5-computer-use-preview-10-2025


## 8. Funciones con `system_instruction`

Las siguientes funciones envuelven la API para incluir un **system prompt** (instrucción de sistema), que define el rol y las reglas del modelo antes de procesar el mensaje del usuario.

| Función | Para qué sirve |
|---|---|
| `chat_with_system` | Un turno sencillo con rol definido |
| `chat_with_system_and_history` | Historial de conversación + rol definido |

In [ ]:
import json, csv, io, time
from google.genai import types


def chat_with_system(system_instruction, user_message, model=None):
    """Genera una respuesta con un rol/reglas definido por system_instruction."""
    m = model or MODELO_FLASH
    config = types.GenerateContentConfig(system_instruction=system_instruction)
    resp = client.models.generate_content(model=m, contents=user_message, config=config)
    return resp.text or ""


def chat_with_system_and_history(system_instruction, history, model=None):
    """Genera una respuesta con system_instruction + historial de conversación."""
    if not history:
        return ""
    m = model or MODELO_FLASH
    config = types.GenerateContentConfig(system_instruction=system_instruction)
    resp = client.models.generate_content(model=m, contents=history, config=config)
    return resp.text or ""


print("✅ chat_with_system y chat_with_system_and_history definidas")

## 9. 🧪 Experimento A — Mismo prompt, 3 system prompts distintos

**Pregunta:** `¿Qué es machine learning?` (igual en los 3 casos)

| System prompt | Qué se espera |
|---|---|
| **A — vago** | Respuesta genérica, longitud impredecible |
| **B — técnico** | Respuesta estructurada, máximo 3 párrafos |
| **C — JSON** | Objeto JSON con campos fijos |

Al finalizar se compara la cantidad de palabras y se verifica si C produce JSON válido.

In [ ]:
PREGUNTA_A = "¿Qué es machine learning?"

SYSTEMS_A = {
    "A — vago": "Eres un asistente útil.",
    "B — técnico": (
        "Eres un profesor universitario de ingeniería de sistemas. "
        "Responde de forma técnica y precisa en máximo 3 párrafos."
    ),
    "C — JSON": (
        "Eres un asistente técnico. "
        "RESPONDE ÚNICAMENTE con un objeto JSON válido con esta estructura exacta, "
        "sin texto adicional ni markdown: "
        '{"concepto": str, "definicion": str, "ejemplo": str, "nivel": "basico|intermedio|avanzado"}'
    ),
}

inicio_a = time.time()
respuestas_a = {}

for nombre, system_text in SYSTEMS_A.items():
    print(f"\n{'─'*55}")
    print(f"System {nombre}")
    print("─" * 55)
    try:
        resp = chat_with_system(system_text, PREGUNTA_A)
        respuestas_a[nombre] = resp
        print(resp)
    except Exception as e:
        respuestas_a[nombre] = ""
        print(f"[ERROR] {e}")

# ── Análisis comparativo ──────────────────────────────────────
print(f"\n{'='*55}")
print("ANÁLISIS")
print("=" * 55)
longitudes = {k: len(v.split()) for k, v in respuestas_a.items()}
for nombre, palabras in longitudes.items():
    print(f"  System {nombre}: {palabras} palabras")

resp_c = respuestas_a.get("C — JSON", "")
try:
    json_limpio = (
        resp_c.strip()
        .removeprefix("```json").removeprefix("```")
        .removesuffix("```").strip()
    )
    json.loads(json_limpio)
    print("  ✅ Respuesta C es JSON válido")
except (json.JSONDecodeError, ValueError):
    print("  ❌ Respuesta C NO es JSON válido")

if longitudes:
    mas_larga = max(longitudes, key=longitudes.get)
    mas_corta = min(longitudes, key=longitudes.get)
    print(f"  Más larga : System {mas_larga} ({longitudes[mas_larga]} palabras)")
    print(f"  Más corta : System {mas_corta} ({longitudes[mas_corta]} palabras)")

print(f"\n⏱  Completado en {time.time() - inicio_a:.2f}s")

## 10. 🧪 Experimento B — Mismo system, 3 formatos de salida

**System prompt:** `"Eres un experto en inteligencia artificial."` (igual en los 3 casos)  
**Pregunta base:** `"Explica las 3 aplicaciones más importantes de la IA"`

| Formato pedido | Se valida |
|---|---|
| **1 — Texto libre** | Solo se imprime |
| **2 — JSON** | Se parsea con `json.loads` |
| **3 — CSV** | Se parsea con `csv.reader` |

In [ ]:
SYSTEM_B = "Eres un experto en inteligencia artificial."
PREGUNTA_BASE_B = "Explica las 3 aplicaciones más importantes de la IA"

FORMATOS_B = {
    "1 — Texto libre": PREGUNTA_BASE_B,
    "2 — JSON": (
        f"{PREGUNTA_BASE_B}. "
        "Responde SOLO con JSON: "
        '[{"aplicacion": str, "descripcion": str, "industria": str}]'
    ),
    "3 — CSV": (
        f"{PREGUNTA_BASE_B}. "
        "Responde SOLO con CSV, primera fila son headers: "
        "aplicacion,descripcion,industria"
    ),
}

inicio_b = time.time()

for nombre_formato, mensaje in FORMATOS_B.items():
    print(f"\n{'─'*55}")
    print(f"Formato {nombre_formato}")
    print("─" * 55)
    try:
        resp = chat_with_system(SYSTEM_B, mensaje)
        print(resp)

        if nombre_formato.startswith("2"):
            try:
                texto_limpio = (
                    resp.strip()
                    .removeprefix("```json").removeprefix("```")
                    .removesuffix("```").strip()
                )
                datos = json.loads(texto_limpio)
                print(f"  ✅ JSON válido — {len(datos)} elemento(s) en la lista")
            except (json.JSONDecodeError, ValueError):
                print("  ❌ La respuesta no es JSON parseable")

        elif nombre_formato.startswith("3"):
            try:
                reader = csv.reader(io.StringIO(resp.strip()))
                filas = list(reader)
                print("  ✅ CSV parseado — filas:")
                for fila in filas:
                    print(f"    {fila}")
            except Exception as csv_err:
                print(f"  ❌ No se pudo parsear el CSV: {csv_err}")

    except Exception as e:
        print(f"[ERROR] {e}")

print(f"\n⏱  Completado en {time.time() - inicio_b:.2f}s")

## 11. 🧪 Experimento C — Memoria: sin historial vs con historial

Se envían dos mensajes:
1. `"Mi nombre es Carlos y soy de Barranquilla"`
2. `"¿Cómo me llamo y de dónde soy?"`

| Variante | Cómo se llama a la API | Resultado esperado |
|---|---|---|
| **Sin historial** | Dos llamadas independientes | El modelo no puede saber el nombre |
| **Con historial** | Una sola petición con todos los turnos | El modelo puede responder correctamente |

> 💡 Este es el mecanismo fundamental detrás de los chatbots con memoria.

In [ ]:
MSG_1 = "Mi nombre es Carlos y soy de Barranquilla"
MSG_2 = "¿Cómo me llamo y de dónde soy?"

inicio_c = time.time()

# ── Sin historial: dos llamadas independientes ────────────────
print("📭 SIN HISTORIAL (llamadas independientes):")
print("─" * 55)
# Llamada 1: el modelo recibe la presentación pero la respuesta se descarta
_ = client.models.generate_content(model=MODELO_FLASH, contents=MSG_1)
# Llamada 2: nueva sesión vacía — el modelo no tiene contexto previo
resp_sin = client.models.generate_content(model=MODELO_FLASH, contents=MSG_2)
print(resp_sin.text)

# ── Con historial: todos los turnos en una sola petición ──────
print("\n📬 CON HISTORIAL (historial acumulado):")
print("─" * 55)
historial_c = [
    {"role": "user",  "parts": [{"text": MSG_1}]},
    {"role": "model", "parts": [{"text": "¡Hola Carlos! Encantado de conocerte."}]},
    {"role": "user",  "parts": [{"text": MSG_2}]},
]
resp_con = client.models.generate_content(model=MODELO_FLASH, contents=historial_c)
print(resp_con.text)

print(
    "\n💡 ¿Por qué son diferentes?\n"
    "  — Sin historial: cada llamada es una sesión nueva. El modelo no tiene\n"
    "    ningún contexto de lo que se dijo antes.\n"
    "  — Con historial: se envían todos los turnos en una misma petición.\n"
    "    El modelo puede 'leer' la conversación completa y responder con contexto.\n"
    "  Esta es la base del manejo de memoria en chatbots conversacionales."
)

print(f"\n⏱  Completado en {time.time() - inicio_c:.2f}s")